In [ ]:
# ============================================================
# Phase 1.1.a — declare tools and target. Nothing is loaded yet.
# ============================================================
# We are not running the model here. We are only stating:
#   - which library we will use (transformers)
#   - which model we will eventually load (Llama 3.2 1B Instruct)
# This separation matters because the *next* chunks will each
# do exactly one thing: load the tokenizer, then load the model.

import torch                                                          # <-- NEW
# torch is the tensor + neural-network library underneath everything.
# Every weight, every input, every intermediate value in this notebook
# will be a torch.Tensor. We rarely call torch directly in Phase 1,
# but we need it imported because the model's weights live as torch tensors.

from transformers import AutoTokenizer, AutoModelForCausalLM          # <-- NEW
# transformers is HuggingFace's library. Two classes are imported:
#   AutoTokenizer       -> picks the right tokenizer for the model ID.
#                          A tokenizer converts text <-> integer IDs.
#   AutoModelForCausalLM -> picks the right model class for causal LM.
#                          "Causal" = predicts the next token given previous.
#                          (Llama, GPT, Mistral are all causal LMs.)
# The "Auto" prefix means: look up the correct concrete class from the
# model's config on HuggingFace Hub. We do not need to hardcode
# "LlamaTokenizer" or "LlamaForCausalLM" ourselves.

MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"                         # <-- NEW
# This is the HuggingFace Hub identifier. Format: "<org>/<model_name>".
# The "-Instruct" suffix means it has been fine-tuned to follow
# instructions (vs. the base model which only continues text).
# This model is gated — you must be logged in via `hf auth login`
# AND have accepted Meta's license on the model's HF page.

print("torch version       :", torch.__version__)
print("model we will load  :", MODEL_ID)
# These prints are diagnostic only. They confirm the cell ran and
# show the torch version (useful when debugging dtype or device issues
# later — bf16 support depends on the torch + CPU combination).

In [ ]:
# ============================================================
# Phase 1.1.b — load the tokenizer (text <-> integer IDs).
# ============================================================
# The tokenizer is a separate artifact from the model weights.
# It is small (a few MB) and loads fast. We load it first because:
#   1. It proves the network and HF auth are working before we
#      commit to the multi-GB model download.
#   2. The tokenizer is what we will use in Phase 1.5 to turn
#      input text into the integer IDs the model expects.

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"

print("torch version       :", torch.__version__)
print("model we will load  :", MODEL_ID)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)                   # <-- NEW
# from_pretrained() does three things:
#   1. Looks up the model's config on HuggingFace Hub.
#   2. Downloads the tokenizer files (vocab, merges, special tokens
#      config) — or reads them from local cache if previously downloaded.
#      Cache lives at ~/.cache/huggingface/hub by default.
#   3. Constructs the correct tokenizer subclass (here:
#      PreTrainedTokenizerFast, which is a Rust-backed BPE tokenizer).
# The returned object is callable: `tokenizer("some text")` returns IDs.

print("tokenizer type:", type(tokenizer).__name__)                    # <-- NEW
# We print the *concrete* class name, not "AutoTokenizer".
# AutoTokenizer is just a factory — the real object is something like
# PreTrainedTokenizerFast. Knowing the concrete type matters when
# you read the HuggingFace docs for that class's methods.

print(tokenizer)                                                      # <-- NEW
# Printing the tokenizer dumps its repr: vocab size, model_max_length,
# the special tokens (bos, eos, pad, unk), and which tokens are
# considered "added" vs from the original vocab. This is a free
# preview of values we will formally inspect in step 1.2.

In [ ]:
# ============================================================
# Phase 1.1.c — sanity check: tokenizer turns text into integer IDs.
# ============================================================
# We are not *using* these IDs yet — that is step 1.5 onwards.
# We only confirm the tokenizer is a working, callable object.
# This is a 30-second test that catches problems early
# (bad auth, wrong model ID, corrupted cache).

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"

print("torch version       :", torch.__version__)
print("model we will load  :", MODEL_ID)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("tokenizer type:", type(tokenizer).__name__)
print(tokenizer)

ids = tokenizer("The capital of France is", return_tensors="pt").input_ids   # <-- NEW
# Calling the tokenizer like a function runs the BPE algorithm:
#   "The capital of France is" -> list of integer token IDs.
# return_tensors="pt" means: return PyTorch tensors (not Python lists
# or numpy arrays). The "pt" stands for PyTorch.
# The returned object is a BatchEncoding (dict-like). We pull out
# `.input_ids` — the actual ID tensor. There are other fields like
# `attention_mask` we will not need yet.
# Note the shape will be [1, N_tok]: the leading 1 is the batch
# dimension (batch of 1 sentence), N_tok is the number of tokens
# the BPE algorithm produced for this string.

print("input_ids:", ids)                                              # <-- NEW
# Prints something like: tensor([[128000, 791, 6864, 315, 9822, 374]])
# Token 128000 is the BOS (beginning-of-sequence) token Llama prepends.
# The remaining IDs are the actual word/subword pieces.

print("shape    :", ids.shape)                                        # <-- NEW
# torch.Size([1, 6]) for this input. The 6 will vary with input length;
# the leading 1 (batch size) is constant for our one-sentence-at-a-time
# usage. Decision #16 in the cont

In [ ]:
# ============================================================
# Phase 1.1.d — load the model weights into RAM. SLOW STEP.
# ============================================================
# This is the only chunk in step 1.1 that does real work.
# First run: downloads ~2.5 GB of weight files (.safetensors shards)
#            from HuggingFace. Takes a few minutes on a normal connection.
# Later runs: reads from local cache (~/.cache/huggingface/hub).
#             Takes ~10–30 seconds — bottleneck is reading bf16 weights
#             from disk into RAM and reconstructing the module tree.
# After this call returns, ~2.4 GB of bf16 weight tensors live in RAM
# (1.24B params × 2 bytes/param). On a t3.xlarge (16 GB RAM) this
# leaves comfortable headroom for activations.

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"

print("torch version       :", torch.__version__)
print("model we will load  :", MODEL_ID)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("tokenizer type:", type(tokenizer).__name__)
print(tokenizer)

ids = tokenizer("The capital of France is", return_tensors="pt").input_ids
print("input_ids:", ids)
print("shape    :", ids.shape)

model = AutoModelForCausalLM.from_pretrained(                         # <-- NEW
    MODEL_ID,                                                         # <-- NEW
    torch_dtype=torch.bfloat16,                                       # <-- NEW
    # Decision #12 from the contract: load weights as bf16, not fp32.
    # Why bf16:
    #   - Halves the RAM footprint (2 bytes/param vs 4).
    #   - bf16 keeps fp32's exponent range (just less mantissa precision),
    #     so it does not underflow/overflow on transformer activations
    #     the way fp16 sometimes does.
    #   - Modern CPUs (and all GPUs) handle bf16 natively or near-natively.
    # On CPU with t3.xlarge, fp32 would also fit (~5 GB) but is wasteful.
    device_map="cpu",                                                 # <-- NEW
    # Decision #3: CPU only. device_map controls *where* each layer's
    # weights are placed. "cpu" forces everything to CPU RAM.
    # On a GPU box we would use "auto" or "cuda" instead.
    # Speed is not the goal of this learning project — clarity is.
)                                                                     # <-- NEW

print("loaded:", type(model).__name__)                                # <-- NEW
# Should print: "loaded: LlamaForCausalLM".
# Like the tokenizer, AutoModelForCausalLM is a factory — the concrete
# class for this checkpoint is LlamaForCausalLM.
# This object is a torch.nn.Module — a tree of submodules with
# learnable parameters. We will explore the tree in chunk 6.

In [ ]:
# ============================================================
# Phase 1.1.e — switch model from training mode to inference mode.
# ============================================================
# Every torch.nn.Module has a `training` flag, default True.
# Some layers behave differently based on this flag:
#   - Dropout: drops random activations during training, no-op in eval.
#   - BatchNorm: uses batch statistics during training, running
#                statistics during eval. (Llama uses RMSNorm, no
#                running stats — but the habit still applies.)
# For inference we MUST call .eval() to disable dropout and any other
# train-only behavior. Forgetting this is a classic source of
# nondeterministic / wrong outputs.
# .eval() returns the model itself (for chaining), and recursively
# sets training=False on every submodule.

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"

print("torch version       :", torch.__version__)
print("model we will load  :", MODEL_ID)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("tokenizer type:", type(tokenizer).__name__)
print(tokenizer)

ids = tokenizer("The capital of France is", return_tensors="pt").input_ids
print("input_ids:", ids)
print("shape    :", ids.shape)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="cpu",
)

print("loaded:", type(model).__name__)

model.eval()                                                          # <-- NEW
# Flips model.training from True to False on the root module
# AND on every submodule (recursive). Side-effect-only; we do not
# capture the return value.

print("training mode:", model.training)   # expect: False             # <-- NEW
# The observable: this prints False. If it ever prints True after
# .eval(), something is wrong (likely a custom subclass overriding
# the flag). For Llama this is reliable.

In [ ]:
# ============================================================
# Phase 1.1.f — the loaded object is a nested tree of modules.
# ============================================================
# This is the single most important observation in step 1.1.
# `model` is not a function and not a dict. It is a torch.nn.Module
# whose children are themselves nn.Modules, recursively, all the way
# down to the matrix multiplications.
# We only look at the top level here. The full walk happens in 1.4.
# Why this matters: every reference in the rest of this project will
# be a path through this tree, e.g.
#   model.model.layers[5].self_attn.q_proj
# That path is just attribute access on nested modules — no magic.

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"

print("torch version       :", torch.__version__)
print("model we will load  :", MODEL_ID)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("tokenizer type:", type(tokenizer).__name__)
print(tokenizer)

ids = tokenizer("The capital of France is", return_tensors="pt").input_ids
print("input_ids:", ids)
print("shape    :", ids.shape)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="cpu",
)

print("loaded:", type(model).__name__)

model.eval()
print("training mode:", model.training)

print("class:", type(model).__name__)                                 # <-- NEW
# Reconfirms the root class. LlamaForCausalLM = "the LM head wrapper
# around the core transformer." It has exactly two children:
#   - model      : the transformer core (embeddings + layers + final norm)
#   - lm_head    : the projection from hidden state -> vocabulary logits

print("top-level children:")                                          # <-- NEW
for name, _child in model.named_children():                           # <-- NEW
    print(" -", name)                                                 # <-- NEW
# named_children() yields (name, submodule) pairs for direct children
# only — NOT recursive. (named_modules() would be recursive.)
# Expected output:
#    - model
#    - lm_head
# Note the slightly confusing naming: `model.model` is the inner
# transformer core; `model` (the outer one) is the LM-head wrapper.
# This is HuggingFace convention, not a typo. Phases 3–7 will open
# `model.model` further (embed_tokens, layers, norm).

In [ ]:
# ============================================================
# Phase 1.1.g — final check: both objects loaded, model in eval mode.
# ============================================================
# An assertion block is the contract for "this step is done."
# If every assert passes, step 1.1 is complete and we can move
# on to 1.2 (inspect model.config) with confidence that the
# foundation is sound.
# We deliberately do NOT assert anything about parameter counts,
# dtypes, or device — those are step 1.2's job, after the learner
# has been taught what to expect from the config.

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"

print("torch version       :", torch.__version__)
print("model we will load  :", MODEL_ID)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("tokenizer type:", type(tokenizer).__name__)
print(tokenizer)

ids = tokenizer("The capital of France is", return_tensors="pt").input_ids
print("input_ids:", ids)
print("shape    :", ids.shape)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="cpu",
)

print("loaded:", type(model).__name__)

model.eval()
print("training mode:", model.training)

print("class:", type(model).__name__)
print("top-level children:")
for name, _child in model.named_children():
    print(" -", name)

assert tokenizer is not None                                          # <-- NEW
# Trivial check — from_pretrained() raises on failure rather than
# returning None, so this is mostly documentation. Still, an explicit
# assert makes the contract visible to anyone reading the notebook later.

assert model is not None                                              # <-- NEW
# Same as above for the model object.

assert model.training is False                                        # <-- NEW
# This is the *real* assertion. If we forget .eval() somewhere, this
# catches it. Inference results would otherwise be subtly wrong
# (dropout active, etc.).

print("[OK] Phase 1.1 complete — tokenizer + model loaded, eval mode on")  # <-- NEW
# A final success line is a Jupyter habit worth keeping. When you
# scroll back to this cell weeks later, you can confirm at a glance
# that this phase ran cleanly without re-reading the asserts.